# π0 (3.3B) - Proprioceptive State Utilization Analysis

**Model**: Physical-Intelligence π0 (3.3B parameters)  
**State Encoder**: `proprio_encoder` - multi-layer MLP (separate from VL backbone)

## Hypothesis
**Proprioceptive state information is underutilized** - the model may not be meaningfully using robot state despite having a dedicated multi-layer encoder.

## Experimental Design
1. **Baseline**: Run normal state through proprio_encoder → capture gradients
2. **Ablation**: Zero out robot_state → capture gradients  
3. **Compare**: Calculate gradient change percentage
4. **Verdict**:
   - <10% change = ❌ UNDERUTILIZED (state doesn't matter)
   - 10-30% change = ⚠️ PARTIALLY UTILIZED
   - >30% change = ✅ WELL UTILIZED (model relies on state)

## Key Metrics
- **Gradient norm on proprio_encoder layers**
- **Gradient change % when state is ablated**

---

## 1. Environment Setup

In [1]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab")
    !nvidia-smi
else:
    print("💻 Running locally")

🚀 Running on Google Colab
Sun Feb 15 04:19:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   49C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
# Install dependencies
!pip install -q torch torchvision transformers accelerate diffusers pillow numpy matplotlib seaborn scikit-learn

print("✅ Dependencies installed")

✅ Dependencies installed


In [3]:
# Clone repository
if IN_COLAB:
    !git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git
    %cd EmbodiedLLM
    !git checkout AnalyseMultipleHooks
    %cd MultipleHooksStudy
else:
    import os
    if not os.path.exists('hooks'):
        print("⚠️ Please run from MultipleHooksStudy directory")
    else:
        print("✅ Hook files found")

Cloning into 'EmbodiedLLM'...
remote: Enumerating objects: 528, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 528 (delta 43), reused 104 (delta 35), pack-reused 412 (from 2)
Receiving objects: 100% (528/528), 141.86 MiB | 15.10 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/EmbodiedLLM
Branch 'AnalyseMultipleHooks' set up to track remote branch 'AnalyseMultipleHooks' from 'origin'.
Switched to a new branch 'AnalyseMultipleHooks'
/content/EmbodiedLLM/MultipleHooksStudy


## 2. Import Hook Framework

In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add hooks to path
sys.path.insert(0, str(Path.cwd() / 'hooks'))

from hooks.model_specific.pi0_hooks import Pi0Hooks

print("✅ π0 hook framework imported")

✅ π0 hook framework imported


## 3. Load π0 Model

**Note**: π0 now supports PyTorch! You need to convert JAX checkpoints first. See: [PyTorch Support docs](https://github.com/Physical-Intelligence/openpi#pytorch-support)

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load real π0 model
print("Loading real π0 model from Physical Intelligence...")

# Install openpi if not already installed
try:
    import openpi
except ImportError:
    print("Installing openpi package...")
    !pip install -q git+https://github.com/Physical-Intelligence/openpi.git
    import openpi

from openpi.training import config as pi0_config
from openpi.policies import policy_config

# Get checkpoint directory
import os
checkpoint_dir = os.environ.get('PI0_CHECKPOINT_DIR', './pi0_checkpoints')

if not os.path.exists(checkpoint_dir):
    print(f"\n⚠️ Checkpoint directory not found: {checkpoint_dir}")
    print("\nTo use π0:")
    print("1. Download checkpoints from: https://huggingface.co/physical-intelligence/pi0")
    print("2. Convert to PyTorch: python -m openpi.tools.convert_jax_to_pytorch")
    print("3. Set PI0_CHECKPOINT_DIR=/path/to/checkpoints")
    raise FileNotFoundError(f"Checkpoints not found at {checkpoint_dir}")

# Load configuration
config = pi0_config.get_config("pi0_base")

# Create trained policy
print(f"Loading from checkpoint: {checkpoint_dir}")
policy = policy_config.create_trained_policy(config, checkpoint_dir)
model = policy.model

# Move to device
model = model.to(device)
if device.type == 'cuda':
    model = model.half()

print("✅ π0 model loaded successfully!")
print(f"   Model: Physical Intelligence π0 (3.3B parameters)")
print(f"   Checkpoint: {checkpoint_dir}")
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Using device: cuda
Loading real π0 model from Physical Intelligence...
Installing openpi package...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.3 MB/s eta 0:00:00


## 4. Discover Model Structure

**Critical**: Verify separate multi-layer proprio encoder!

In [ ]:
# Initialize hook manager
hook_manager = Pi0Hooks(model)

# Discover structure
structure = hook_manager.discover_model_structure()

print("\n" + "="*60)
print("π0 Model Structure Discovery")
print("="*60)
print(f"Model Name: {structure['model_name']}")
print(f"Has Proprio Encoder: {structure['has_proprio_encoder']}")
print(f"Proprio Encoder Type: {structure['proprio_encoder_type']}")
print(f"Num Proprio Encoder Layers: {structure.get('proprio_encoder_layers', 0)}")

print(f"\nComponents Found:")
for component, attr in structure['components'].items():
    print(f"  - {component}: {attr}")

# Highlight separate encoder
if structure.get('proprio_encoder_layers', 0) > 1:
    print(f"\n🎯 Separate Multi-Layer Proprio Encoder VERIFIED!")
    print(f"   Layers: {structure['proprio_encoder_layers']}")
    print(f"   This is π0's key advantage over simpler encoders")
else:
    print("\n⚠️ Multi-layer encoder not found - check model structure")

print("="*60)

## 5. Attach Analysis Hooks

In [ ]:
# Attach gradient hooks only (focus on proprio_encoder)
print("Attaching gradient hooks to proprio_encoder layers...")
hook_manager.attach_gradient_hooks()
print("✓ Gradient hooks attached to multi-layer proprio_encoder")
print("\nWill compare gradients: Normal state vs Zero state")

## 6. Prepare Sample Data

In [ ]:
from PIL import Image

# Sample data
sample_image = Image.new('RGB', (224, 224), color='blue')
sample_instruction = "Fold the blue towel in half and place it on the table."
sample_state = torch.randn(1, 7).to(device).half()  # 7-DOF robot state

# Prepare inputs
inputs = {
    'pixel_values': torch.randn(1, 3, 224, 224).to(device).half(),
    'input_ids': torch.randint(0, 50000, (1, 20)).to(device),
    'robot_state': sample_state
}

print("✅ Sample data prepared")
print(f"Image: {sample_image.size}")
print(f"Instruction: {sample_instruction}")
print(f"Robot state shape: {sample_state.shape}")

## 7. Run Forward and Backward Pass

In [ ]:
# Set to training mode
model.train()

# Forward pass
print("Running forward pass...")
outputs = model(**inputs)

# Compute loss
loss = outputs.mean()

print("Running backward pass...")
loss.backward()

print("✅ Forward and backward pass completed")
print(f"Loss: {loss.item():.4f}")
print(f"Action prediction shape: {outputs.shape}")
print(f"Action range: [{outputs.min().item():.4f}, {outputs.max().item():.4f}]")

## 8. Baseline: Capture Gradients with Normal State

In [ ]:
# Capture proprio_encoder gradients with NORMAL state
hook_manager.reset()
results_baseline = hook_manager.get_results()
layer_profiles = results_baseline.get('gradient_flow', {}).get('layer_profiles', {})

# Extract proprio_encoder gradient
baseline_proprio_grad = layer_profiles.get('proprio_encoder', {})
baseline_norm = baseline_proprio_grad.get('gradient_norm', 0.0)

print("Proprio Encoder Gradients (NORMAL STATE)")
print(f"Gradient norm: {baseline_norm:.6f}")
print("\nThis is our baseline - model sees real robot state")

## 9. Ablation: Zero Out State Encoder Output

**Better Ablation Strategy**: Zero the OUTPUT of `proprio_encoder`, not the input. 

This directly tests: "What if the multi-layer encoder contributed nothing?"

In [ ]:
# Create hook to zero out proprio_encoder's output
ablation_handle = None

def zero_output_hook(module, input, output):
    """Replace proprio_encoder output with zeros"""
    return torch.zeros_like(output)

# Find and hook the proprio_encoder
for name, module in model.named_modules():
    if 'proprio_encoder' in name and 'layer' not in name:  # Main encoder, not sub-layers
        ablation_handle = module.register_forward_hook(zero_output_hook)
        print(f"✓ Attached ablation hook to: {name}")
        break

print("\nRunning ablation (proprio_encoder OUTPUT = zeros)...")
hook_manager.reset()
model.zero_grad()

# Forward + backward with same inputs, but proprio_encoder outputs zeros
outputs_ablated = model(**inputs)
loss_ablated = outputs_ablated.mean()
loss_ablated.backward()

# Remove ablation hook
if ablation_handle:
    ablation_handle.remove()

print("✓ Ablation complete - multi-layer encoder contribution removed")

# Capture gradients
results_ablated = hook_manager.get_results()

## 10. Compare Gradients: Normal vs Ablation

In [ ]:
# Extract proprio_encoder gradients from ablation run
layer_profiles_ablated = results_ablated.get('gradient_flow', {}).get('layer_profiles', {})
ablated_proprio_grad = layer_profiles_ablated.get('proprio_encoder', {})
ablated_norm = ablated_proprio_grad.get('gradient_norm', 0.0)

print("Proprio Encoder Gradients (ZERO STATE)")
print(f"Gradient norm: {ablated_norm:.6f}\n")

# Calculate change
grad_change_pct = abs(baseline_norm - ablated_norm) / baseline_norm * 100 if baseline_norm > 0 else 0

print(f"{'='*60}")
print(f"COMPARISON")
print(f"{'='*60}")
print(f"Normal State Gradient:   {baseline_norm:.6f}")
print(f"Ablation Gradient:       {ablated_norm:.6f}")
print(f"Change:                  {grad_change_pct:.1f}%")
print(f"{'='*60}")

## 11. Verdict: Is Proprioceptive State Utilized?

In [ ]:
# Verdict thresholds
if grad_change_pct < 10:
    verdict = "❌ UNDERUTILIZED"
    explanation = "When proprio_encoder output is zeroed, gradients barely change. The model doesn't rely on state encoder's contribution."
elif grad_change_pct < 30:
    verdict = "⚠️ PARTIALLY UTILIZED"
    explanation = "Some gradient sensitivity when state encoder is removed, but the dependency is weak."
else:
    verdict = "✅ WELL UTILIZED"
    explanation = "Strong gradient response when state encoder output is ablated. The model meaningfully uses state information."

print(f"\n{'='*60}")
print(f"VERDICT: {verdict}")
print(f"{'='*60}")
print(f"\nGradient Change: {grad_change_pct:.1f}%")
print(f"\n{explanation}")
print(f"\nMethodology: Measured gradient sensitivity when proprio_encoder")
print(f"output was replaced with zeros (output ablation, not input ablation).")
print(f"{'='*60}")

# Export results
results = {
    'model': 'π0 (3.3B)',
    'state_encoder': 'proprio_encoder (multi-layer MLP)',
    'ablation_method': 'output_ablation',
    'baseline_grad_norm': float(baseline_norm),
    'ablated_grad_norm': float(ablated_norm),
    'gradient_change_pct': float(grad_change_pct),
    'verdict': verdict
}

import json
with open('pi0_state_utilization_results.json', 'w') as f:
    json.dump(results, f, indent=2)
    
print(f"\n✓ Results exported to: pi0_state_utilization_results.json")